# Univariate ETT -- what four covariance parameterizations look like

Section 8 builds the paper figure `ts_etth2_forecast_correlation.pdf` (ETTh2, univariate,
$T = 96$, LorD against SCORE, placed next to the EUPPBench figure); sections 4-7 are
diagnostic figures, written only with `SAVE = True`.

Four PatchTST runs per task, differing in how the predictive covariance over the forecast
horizon is parameterized:

| arm | $\Sigma$ | numbers emitted per case |
|---|---|---|
| `LorD (r=1)` | $\mathrm{diag}(v) + uu^\top$ | $2T$ |
| `Chol` | $LL^\top$, $L$ unconstrained lower-triangular | $T(T{+}1)/2$ |
| `SB` | none -- an $M$-member ensemble; $\Sigma$ exists only as a sample covariance | $M \times T$ |
| `SCORE` | $\Sigma = D\,B^{H}\mathrm{diag}(s)\,B\,D$, $B$ the DFT | $2T$ |

The event axis is **time**, so $\Sigma$ is a $T\times T$ matrix over lead times and every
structural question is directly plottable -- which is not the case for the gridded or station
tasks. Five figures per horizon:

1. **predictive panel + estimated covariance**, one row per arm, for a single test window;
2. **sample paths and the mean**, one panel per arm, same window;
3. **the covariance comparison**: each arm's test-set-mean covariance over the empirical
   covariance of *its own* residuals;
4. **the autocorrelation**: every arm's predicted ACF as a solid line against the empirical
   ACF of *its own* residuals as a dashed line in the same colour;
5. **the ACF ratio**: empirical over predicted, again per arm against its own residuals.

**Two arm sets, four sections.** Sections 4-5 take each family at the objective it is
*actually trained with* (maximum likelihood for the two explicit-density Gaussians, a scoring
rule for the other two); sections 6-7 repeat everything with **every arm at the kernel score**,
which is the matched-objective reading. Both are on the same window of the same dataset at
$T = 96$ and $T = 192$.

## 0 Setup

In [ ]:
import os
import sys
from pathlib import Path

# --- Project-root bootstrap: put the repo root on sys.path so `import src...` works, and
# export PROJECT_ROOT for any Hydra ${oc.env:PROJECT_ROOT} lookups. ---
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project-root").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))
if str(PROJECT_ROOT / "notebooks") not in sys.path:   # so `import plot_config` works
    sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import json

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import ticker as mticker
from matplotlib.lines import Line2D

import plot_config as pc

# The run loaders live in `src/evaluation/calibration.py`, shared with the calibration script,
# so the notebooks and the script agree on *which* checkpoint of a run is used.
# `find_checkpoint` prefers `stage2.ckpt` over `model.ckpt` and never `stage1.ckpt`, which is
# what makes SCORE (two-stage fit) comparable with the single-stage arms.
from src.evaluation.calibration import build_model_from_run

PLOT = pc.apply_style()                 # scienceplots + the paper's rcParams
DIVERGING = pc.diverging_cmap()         # signed fields: covariance and correlation
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(sys.executable)
print("project root:", PROJECT_ROOT, "| device:", DEVICE)

## 1 Configuration

**Dataset and horizons.** `DATASET` picks the task; `T_SHORT` / `T_LONG` pick the two horizons
every section is built on.

| dataset | cadence | $T = 96$ is | $T = 192$ is | checkpoints |
|---|---|---|---|---|
| `etth1`, `etth2` | hourly (`ett_hour`) | 4 days | 8 days | in the results tree |
| `ettm1`, `ettm2` | 15-minute (`ett_minute`) | 24 h | 48 h | in the results tree, **no Chol** (not trained on ETTm) |

`ettm1` / `ettm2` are wired up here -- data paths, labels, and the cadence handling every
figure needs. Chol was **not trained on ETTm**, though, and every section below has a Chol
panel, so on ETTm `prepare` stops with `FileNotFoundError` at that arm unless it is dropped
from `ARMS_ASTRAINED` / `ARMS_KERNEL_SCORE`. The ETTm SB sampler also collapses to a point
forecast on **every** seed (`crps/mae` $0.99$-$1.00$), which the `crps/mae` guard below reports.

**Cadence is read from the run, not from the dataset name.** ETTm is 15-minute data, so
`pred_len = 96` is 24 hours there against 96 on ETTh, and every lead-time axis, tick ladder and
lag axis below is in *hours* rather than steps to keep the two comparable. The step length is
taken from the checkpoint's own `datamodule.data_split` via `CADENCE_HOURS`, so it cannot
disagree with the weights being plotted. A non-ETT benchmark needs an entry there before it
will be right: `weather` is 10-minute data despite carrying `freq: h`.

**Seed.** `SEED = 1`, the lowest of seeds 1-5. On ETTh no univariate SB seed collapses
to a point forecast at either horizon (`crps/mae` $0.73$-$0.79$); ETTm is the exception, see
above. `prepare` prints the `crps/mae` collapse detector for the selected seed, so a seed
switch cannot land on a collapsed run silently.

**Window.** `SAMPLE_INDEX` indexes the *test split of that horizon*. Both horizons share
`seq_len = 336` and the same chronological split, and window $i$ starts at the same absolute
timestamp at either horizon, so one index gives genuinely the same forecast origin in both
sections.

**Draws.** `N_DRAW` is how many members the SB backbone is asked for; the closed-form
families ignore it, and it is also what its sample covariance and marginal std are estimated
from. It is deliberately far above the protocol's `n_eval_samples: 100` -- a sample covariance
from 100 members is rank-99 and visibly noisy at $T = 192$, and the point of these panels is
what the sampler's dependence *is*, not what a 100-member estimate of it looks like.

In [ ]:
DATASET = "etth1"        # "etth1" | "etth2" | "ettm1" | "ettm2" (see the note on ETTm above)
T_SHORT, T_LONG = 96, 192
SEED = 1                 # see the note above before changing this
MODE = "univariate"      # the results tree's <dataset>/<mode>/T<pred_len> level

from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

#: The datamodule is always instantiated with an explicit local path rather than the config's
#: own, because a run's recorded `data_path` can point to the machine it was trained on.
DATA_PATHS = {
    "etth1": str(configured_path("time_series_dir") / "ETT-small/ETTh1.csv"),
    "etth2": str(configured_path("time_series_dir") / "ETT-small/ETTh2.csv"),
    "ettm1": str(configured_path("time_series_dir") / "ETT-small/ETTm1.csv"),
    "ettm2": str(configured_path("time_series_dir") / "ETT-small/ETTm2.csv"),
}
DATASET_LABEL = {"etth1": "ETTh1", "etth2": "ETTh2", "ettm1": "ETTm1", "ettm2": "ETTm2"}

#: Hours per forecast step, keyed on the run's own `datamodule.data_split`. This is what puts
#: the hourly and the 15-minute benchmarks on one physical axis; an unlisted split falls back
#: to 1 h with a warning rather than silently mislabelling the axis.
CADENCE_HOURS = {"ett_hour": 1.0, "ett_minute": 0.25}

#: **As-trained arms** (sections 4-5), in order of increasing covariance generality and then
#: SCORE: LorD (r=1) and Chol (the unstructured ceiling) at maximum likelihood, then SB and
#: SCORE at the kernel score. Every label
#: carries its objective, because with a mixed grid a bare arm name is ambiguous. Colors are
#: `plot_config`'s semantic roles, matching `runtime_analysis.ipynb`'s arm map.
ARMS_ASTRAINED = {
    "LorD (r=1, NLL)": dict(
        slug="phys_lora_r1_nll",
        label=r"LorD $r{=}1$ (NLL)", color=pc.COLORS["lora"]),
    "Chol (NLL)": dict(
        slug="phys_cholesky_nll",
        label="Chol (NLL)", color=pc.COLORS["log_score"]),
    "SB (KS)": dict(
        slug="generative_kernel_score",
        label="SB (KS)", color=pc.COLORS["energy_score"]),
    "SCORE (KS)": dict(
        slug="whitened_spectral_diag_s1crps_kernel_score",
        label="SCORE (KS)", color=pc.COLORS["kernel_score"]),
}

#: **Matched-objective arms** (sections 6-7): the same four parameterizations, every one fitted
#: with the fair Gaussian kernel score. Same order, same colors, so a figure here is directly
#: comparable panel-for-panel with its section 4-5 counterpart. `SCORE` is the identical
#: checkpoint in both sets -- it has only ever been kernel-score fitted -- which makes it the
#: fixed point the other three move against.
ARMS_KERNEL_SCORE = {
    "LorD (r=1, KS)": dict(
        slug="phys_lora_r1_kernel_score",
        label=r"LorD $r{=}1$ (KS)", color=pc.COLORS["lora"]),
    "Chol (KS)": dict(
        slug="phys_cholesky_kernel_score",
        label="Chol (KS)", color=pc.COLORS["log_score"]),
    "SB (KS)": dict(
        slug="generative_kernel_score",
        label="SB (KS)", color=pc.COLORS["energy_score"]),
    "SCORE (KS)": dict(
        slug="whitened_spectral_diag_s1crps_kernel_score",
        label="SCORE (KS)", color=pc.COLORS["kernel_score"]),
}

SAMPLE_INDEX = 400       # which test window the single-window figures show
N_DRAW = 1000            # members asked of the SB backbone (see above)
N_SHOW = 10              # sample paths kept per arm (figure 2 can show fewer, not more)
DENORMALIZE = True       # undo the dataset standardisation -> degC
CONTEXT_SHOW = 168       # steps of input window shown; None = all 336
EVAL_BATCH_SIZE = 64     # ample for Chol at T=192
MAX_TEST_BATCHES = None  # None = the whole test split; an int truncates (debug)

#: Longest lag drawn in figures 4-5, as a fraction of the horizon. The diagonal average at
#: offset h rests on T-h pairs, so the far tail of a lag plot is one or two numbers wide and
#: says nothing; `src/evaluation/calibration.py` caps its own ladder at 0.25 for the same
#: reason. Half the horizon is generous by that standard and still cuts the useless part.
LAG_MAX_FRAC = 0.5

#: In figure 5's ratio panel, lags where the *predicted* correlation is below this are left
#: blank: the ratio is a division by ~0 there and the panel would be all spikes.
RATIO_MIN_RHO = 0.05

SAVE = False             # True -> every figure lands in notebooks/figures/ at paper size
FIG_PREFIX = f"ts_{DATASET}_{MODE}"


def save(fig, name):
    """Write `fig` to notebooks/figures/<FIG_PREFIX>_<name>.pdf when SAVE is on.

    Off by default: these are diagnostics, and a notebook re-run should not quietly rewrite
    the paper's figure directory.
    """
    if SAVE:
        print("saved:", pc.save(fig, f"{FIG_PREFIX}_{name}.pdf"))

## 2 Loading a run, and the covariance it implies

**Rebuilding.** A checkpoint stores only `backbone.*` weights -- the predictive method is not
an `nn.Module` -- so both the method and the backbone are re-instantiated from the run's own
`config.yaml` before the state dict is loaded. `build_model_from_run` does exactly that, and
handles the two things that are easy to get wrong: the kernel bandwidths are *derived* at
runtime (`gamma = sqrt(gamma_* * d)`), and `PatchTST` needs `group_denorm`, `other_zero_init`
and `noise` from the method, because its RevIN layer denormalizes each head group by its own
rule and the SB head needs its noise widths.

**The covariance.** No family materializes $\Sigma$ -- it would be 4.6 GiB per case at ERA5
scale, and SB has no $\Sigma$ at all -- so there is no `get_cov`. At
$V = T \le 720$ we can rebuild it from the two accessors *every* family implements, through the
identity the dense masked scores already use (`PredictiveMethod._dense_score_terms`):

$$\Sigma_{ij} = \tfrac12\bigl(\Sigma_{ii} + \Sigma_{jj} - \mathrm{Var}(X_i - X_j)\bigr),$$

with $\Sigma_{ii}$ from `get_marg_std` and $\mathrm{Var}(X_i - X_j)$ from `pair_diff_var`. This
is exact, and it is **family-agnostic**: it goes through the same closed forms the scores are
computed from, so no panel here can drift from what was optimized. For SB the
same two accessors are ensemble second moments, so the identity returns its *sample*
covariance -- the only $\Sigma$ that arm has.

Note what this deliberately does *not* use: SCORE overrides `_dense_score_terms` to score
its correlation core on the **standardized target**, so reading that path would give the core
rather than the physical covariance. Going through `get_marg_std` and
`pair_diff_var` returns the physical $\Sigma$ for every family, which is what is comparable
across them and what is in degC.

**The empirical side.** Every arm's residual covariance is computed from **its own** residuals
$r = y - \mu_{\text{arm}}$, never from a shared mean. That is the only definition under which
"predicted against realised" is a statement about that arm: a different mean leaves a
differently-correlated residual, and on the NLL-fitted arms the means are visibly flatter, so
substituting one arm's residuals for another's is not a neutral simplification. Figures 3, 4
and 5 are all per-arm for this reason -- each of them puts a model quantity next to the
realised quantity of the *same* arm.

That does raise the question of whether the arms are still comparable with each other, since
each is judged against a slightly different reference. They are, and figure 4 shows it
directly: the dashed empirical curves nearly coincide across arms, so the spread between the
solid predicted curves is a difference between models rather than an artefact of four
different means. Where the dashed lines *do* separate, that separation is itself the finding,
and it is about the means.

Everything is reported in **degC** by undoing the train-range standardisation: a mean shifts
and scales, a std scales, a covariance scales by the square.

In [ ]:
_MODEL_CACHE = {}


def run_dir(dataset, T, slug, seed=None, mode=None):
    """`<shared>/time_series/<dataset>/<mode>/T<pred_len>/<slug>/seed<N>` -- the path *is* the
    run identity in this tree, so nothing else needs to be recorded here."""
    return (SHARED / "time_series" / dataset / (MODE if mode is None else mode)
            / f"T{T}" / slug / f"seed{SEED if seed is None else seed}")


def load_model(dataset, T, slug, seed=None):
    """`(model, method, datamodule, cfg, ckpt)` for one arm, cached.

    Cached because the Cholesky head is the heaviest checkpoint to load (1.4 GB at the retired
    T=720): it emits one row of `L` per lead time, i.e. `T x (T+1)` numbers per case. With both arm sets and
    both horizons this notebook holds two of them; call `_MODEL_CACHE.clear()` followed by
    `torch.cuda.empty_cache()` to give that memory back.
    """
    key = (dataset, T, slug, SEED if seed is None else seed)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = build_model_from_run(
            run_dir(dataset, T, slug, seed=key[3]),
            device=DEVICE,
            data_path=DATA_PATHS[dataset],   # the recorded path may not exist here -- see section 1
            num_workers=0,
            eval_batch_size=EVAL_BATCH_SIZE,
            persistent_workers=False,
        )
    return _MODEL_CACHE[key]


def step_hours(cfg):
    """Hours per forecast step, from the run's own split. See `CADENCE_HOURS`."""
    split = str(cfg.datamodule.get("data_split"))
    if split not in CADENCE_HOURS:
        print(f"warning: no CADENCE_HOURS entry for data_split={split!r}; "
              f"assuming 1 h per step, so every time axis below may be mislabelled")
    return CADENCE_HOURS.get(split, 1.0)


def full_covariance(method, pred):
    """Dense `(B, V, V)` predictive covariance from the family's own accessors."""
    var = method.get_marg_std(pred).flatten(start_dim=1).pow(2)      # (B, V)
    rows = torch.arange(var.shape[1], device=var.device)
    pair = method.pair_diff_var(pred, rows)                          # (B, V, V)
    cov = 0.5 * (var.unsqueeze(-1) + var.unsqueeze(-2) - pair)
    return 0.5 * (cov + cov.transpose(-1, -2))   # symmetrize away rounding


def cov_to_corr(cov, eps=1e-12):
    """Correlation matrix; NaN wherever a marginal variance is zero."""
    d = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    denom = np.outer(d, d)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom > eps, cov / np.where(denom > eps, denom, 1.0), np.nan)


def lag_average(mat):
    """Diagonal-averaged `mat`: entry `h` is the mean of its `h`-th offset diagonal.

    For a stationary (Toeplitz) covariance this loses nothing and returns the autocovariance
    function; for a non-stationary one it is the stationary part of it. Applied to a
    correlation matrix it returns the autocorrelation function.
    """
    return np.array([np.nanmean(np.diagonal(mat, offset=h)) for h in range(mat.shape[0])])


def denorm(ds):
    """`(scale, shift)` of the target column, or `(1, 0)` with `DENORMALIZE` off."""
    if not (DENORMALIZE and ds.scale):
        return 1.0, 0.0
    return float(ds.scaler.scale_[-1]), float(ds.scaler.mean_[-1])

In [ ]:
@torch.no_grad()
def analyze_window(dataset, T, arm, index=None, n_draw=None):
    """One test window of one arm: context, truth, mean, marginal std, dense Sigma, draws."""
    model, method, dm, cfg, ckpt = load_model(dataset, T, arm["slug"])
    ds = dm.test_dataset
    index = SAMPLE_INDEX if index is None else index
    x, y = ds[index]

    # `n_samples` is ignored by every closed-form family (`pred` holds parameters and
    # `get_samples` draws from them); for SB it is the member count, fixed
    # here because each member costs a forward pass through the noise injector.
    pred = model(x.unsqueeze(0).to(DEVICE), n_samples=N_DRAW if n_draw is None else n_draw)
    mean = method.get_mean(pred)[0, 0]
    std = method.get_marg_std(pred)[0, 0]
    cov = full_covariance(method, pred)[0]
    draws = method.get_samples(pred, n_samples=N_SHOW)[0, 0]

    scale, shift = denorm(ds)
    return {
        "arm": arm,
        "T": T,
        "index": index,
        "ckpt": ckpt,
        "step_hours": step_hours(cfg),
        "method_cls": cfg.method._target_.split(".")[-1],
        "loss": cfg.method.get("stage2_loss") or cfg.method.get("loss"),
        "context": x[0].cpu().numpy() * scale + shift,           # (seq_len,)
        "target": y[0].cpu().numpy() * scale + shift,            # (T,)
        "mean": mean.cpu().numpy() * scale + shift,              # (T,)
        "std": std.cpu().numpy() * scale,                        # (T,)
        "cov": cov.cpu().numpy() * scale ** 2,                   # (T, T)
        "draws": draws.cpu().numpy() * scale + shift,            # (T, N_SHOW)
        "units": "degC" if scale != 1.0 else "norm.",
    }


@torch.no_grad()
def scan_test(dataset, T, arm, max_batches=None):
    """One pass over the test split: the mean predictive Sigma and the empirical residual one.

    The predictive covariance is input-dependent for every arm here (LorD's `u`, the
    whitened `D`, the Cholesky factor and the sampler's members all move with the window), so
    the *structure* has to be read off an average over cases rather than off one window. The
    empirical residual covariance is the reference it is compared against, and it is this
    arm's own -- see section 2.

    The test loader keeps `drop_last=True` -- the training protocol -- so the last partial
    batch of windows is not scored here either.
    """
    model, method, dm, cfg, ckpt = load_model(dataset, T, arm["slug"])
    scale, _ = denorm(dm.test_dataset)
    max_batches = MAX_TEST_BATCHES if max_batches is None else max_batches

    resid, cov_sum, n = [], None, 0
    for bi, (x, y) in enumerate(dm.test_dataloader()):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)                       # protocol member count for SB
        mu = method.get_mean(pred).flatten(start_dim=1)
        cov = full_covariance(method, pred)
        cov_sum = cov.sum(0) if cov_sum is None else cov_sum + cov.sum(0)
        resid.append((y.flatten(start_dim=1) - mu).cpu())
        n += x.shape[0]

    resid = torch.cat(resid).numpy() * scale
    return {
        "arm": arm,
        "T": T,
        "n": n,
        "step_hours": step_hours(cfg),
        "rmse": float(np.sqrt((resid ** 2).mean())),
        "mean_cov": cov_sum.cpu().numpy() * scale ** 2 / n,      # (T, T) model
        "emp_cov": np.cov(resid, rowvar=False),                  # (T, T) own residuals
        "units": "degC" if scale != 1.0 else "norm.",
    }


def collapse_check(dataset, T, arms, seed=None):
    """`crps/mae` per arm from the synced `metrics.json` -- the posterior-collapse detector.

    For a point forecast CRPS *equals* MAE, so the ratio is an exact test for an ensemble that
    has degenerated to its mean; it needs no re-run and no calibration file. Healthy Gaussian
    arms sit at 0.70-0.75 here, `deterministic_mse` at exactly 1.0, and the three collapsed
    generative seeds of this campaign at 0.988-0.992. The reported kernel score does *not*
    separate them -- the collapsed seeds score slightly better on it -- which is why this is
    checked on CRPS. Collapse leaves sigma at ~1e-3 rather than 0, so no degeneracy guard
    inside the plotting code would catch it; this print is the guard.
    """
    rows = []
    for name, arm in arms.items():
        path = run_dir(dataset, T, arm["slug"], seed=seed) / "metrics.json"
        if not path.exists():
            rows.append((name, None, "no metrics.json"))
            continue
        test = json.loads(path.read_text())["test"]
        ratio = test["crps"] / test["mae"]
        rows.append((name, ratio, "COLLAPSED" if ratio > 0.95 else "ok"))
    width = max(len(n) for n, _, _ in rows)
    print(f"{DATASET_LABEL.get(dataset, dataset)} {MODE} T={T} "
          f"seed{SEED if seed is None else seed}")
    for name, ratio, flag in rows:
        shown = "  --  " if ratio is None else f"{ratio:.3f}"
        print(f"  {name:{width}s}  crps/mae = {shown}  {flag}")
    return rows


def prepare(dataset, T, arms=None):
    """`(windows, scans)` for one horizon and one arm set."""
    arms = ARMS_ASTRAINED if arms is None else arms
    collapse_check(dataset, T, arms)
    windows = {name: analyze_window(dataset, T, arm) for name, arm in arms.items()}
    scans = {name: scan_test(dataset, T, arm) for name, arm in arms.items()}
    width = max(len(n) for n in arms)
    for name in arms:
        w, s = windows[name], scans[name]
        print(f"  {name:{width}s} {w['method_cls']:20s} loss={str(w['loss']):13s} "
              f"n={s['n']:5d}  RMSE={s['rmse']:.3f} {s['units']}  "
              f"window sigma={w['std'].mean():.3f} {w['units']}")
    return windows, scans

## 3 Figure builders

Five builders, each taking the objects section 2 produced and nothing else, so the four
sections below are the same five calls four times over and cannot drift apart. Arm colour and
label travel *inside* those objects, which is what lets one builder serve both arm sets.

**Time axes.** Everything is in hours, `lead * step_hours`, so an ETTh and an ETTm figure are
physically comparable rather than both running 1..T in nameless steps. Ticks come off a
calendar ladder (1, 3, 6, 12, 24, 48, 72 h, then weeks), picking the coarsest that still gives
a few ticks.

**Colour scales.** A covariance panel's scale is set by the largest **off-diagonal** entry and
shared across every panel of the figure: the diagonal is the marginal variance, an order of
magnitude larger, and letting it set the range would flatten all off-diagonal structure to
white. It saturates by design. Correlation panels need no such choice -- they are on
$[-1, 1]$ everywhere.

In [ ]:
#: Calendar-ish tick spacings in hours, coarsest last.
_TICK_LADDER = (1, 3, 6, 12, 24, 48, 72, 168, 336, 720, 1440)


def hour_ticks(span_h, max_ticks=6):
    """Tick positions in hours for an axis spanning `span_h`, off `_TICK_LADDER`."""
    for step in _TICK_LADDER:
        if span_h / step <= max_ticks:
            break
    ticks = [t for t in np.arange(step, span_h + 1e-9, step)]
    return ticks if ticks else [span_h]


def lead_axis(T, step_hours):
    """`(hours, ticks)` for a lead-time axis of `T` steps."""
    hours = np.arange(1, T + 1) * step_hours
    return hours, hour_ticks(float(hours[-1]))


def lag_axis(T, step_hours, max_frac=None):
    """`(lags_in_hours, ticks, n_lags)` for a lag axis, capped at `LAG_MAX_FRAC` of T."""
    frac = LAG_MAX_FRAC if max_frac is None else max_frac
    n = max(2, int(round(frac * T)))
    lags = np.arange(n) * step_hours
    return lags, hour_ticks(float(lags[-1])), n


def offdiag_absmax(mats):
    """Largest |off-diagonal| entry over several matrices -- a shared symmetric colour limit."""
    vals = []
    for m in mats:
        a = np.abs(np.asarray(m)).copy()
        np.fill_diagonal(a, 0.0)
        vals.append(float(np.nanmax(a)))
    v = max(vals)
    return v if v > 0 else 1.0


def matrix_panel(ax, mat, T, step_hours, kind, vmax=None):
    """One lead-time x lead-time heatmap, with the shared conventions of every panel here."""
    hours, ticks = lead_axis(T, step_hours)
    lim = 1.0 if kind == "corr" else vmax
    im = ax.pcolormesh(hours, hours, mat, cmap=DIVERGING, vmin=-lim, vmax=lim,
                       shading="nearest", rasterized=True)
    ax.set_aspect("equal")
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.grid(False)          # gridlines over a heatmap are noise, not structure
    return im


def _title(obj, extra=""):
    """`<Dataset> <mode>, T = <T>` plus whatever the figure wants to add."""
    return (f"{DATASET_LABEL.get(DATASET, DATASET)} {MODE}, $T = {obj['T']}$"
            f"{extra}, seed{SEED}")

### Figure 1 -- predictive panel and estimated covariance

One row per arm. **Left**: the input window, the true future, the predictive mean and the
$\pm 2\sigma$ *marginal* band -- everything a univariate score sees. **Right**: the
$T \times T$ covariance (or correlation, with `kind="corr"`) that the same forward pass
implies for that same window. Two arms can be indistinguishable on the left and completely
different on the right; that gap is the whole point of the figure.

In [ ]:
def plot_predictive_and_cov(windows, kind="cov", context_show=None, row_height=0.30):
    """Figure 1: per-arm forecast panel (left) and estimated Sigma or correlation (right)."""
    names = list(windows)
    first = windows[names[0]]
    T, sh = first["target"].shape[0], first["step_hours"]
    seq_len = first["context"].shape[0]
    show = CONTEXT_SHOW if context_show is None else context_show
    show = seq_len if show is None else min(show, seq_len)
    t_ctx = np.arange(-show, 0) * sh
    t_fut, _ = lead_axis(T, sh)
    mats = {n: (cov_to_corr(w["cov"]) if kind == "corr" else w["cov"])
            for n, w in windows.items()}
    vmax = offdiag_absmax(mats.values())

    # One y range for every row: the rows differ only in the predictive, and a per-row
    # autoscale would rescale exactly the thing the figure is comparing (the band width).
    lo = min(min(float(w["context"][-show:].min()), float(w["target"].min()),
                 float((w["mean"] - 2 * w["std"]).min())) for w in windows.values())
    hi = max(max(float(w["context"][-show:].max()), float(w["target"].max()),
                 float((w["mean"] + 2 * w["std"]).max())) for w in windows.values())
    span = hi - lo

    fig, axes = pc.subplots(len(names), 2, aspect=row_height * len(names),
                            gridspec_kw={"width_ratios": [2.3, 1.0]}, squeeze=False)
    for i, name in enumerate(names):
        w = windows[name]
        color = w["arm"]["color"]

        ax = axes[i, 0]
        ax.plot(t_ctx, w["context"][-show:], color="0.55",
                lw=PLOT.thin_line_width, label="input")
        ax.plot(t_fut, w["target"], color=pc.COLORS["signal"],
                lw=PLOT.thin_line_width, label="truth")
        ax.plot(t_fut, w["mean"], color=color, lw=PLOT.line_width, label="mean")
        ax.fill_between(t_fut, w["mean"] - 2 * w["std"], w["mean"] + 2 * w["std"],
                        color=color, alpha=PLOT.band_alpha, lw=0, label=r"$\pm2\sigma$")
        ax.axvline(0.5 * sh, color=pc.COLORS["guide"], lw=PLOT.thin_line_width, ls=":")
        ax.set_ylim(lo - 0.04 * span, hi + 0.12 * span)   # headroom for the row-0 legend
        ax.set_ylabel(f"{w['arm']['label']}\n[{w['units']}]")
        if i == len(names) - 1:
            ax.set_xlabel("time relative to forecast origin [h]")
        else:
            ax.set_xticklabels([])
        if i == 0:
            ax.legend(loc="upper left", ncols=4, columnspacing=1.0, handlelength=1.4)
            ax.set_title("forecast: input, truth, mean, marginal band")

        ax = axes[i, 1]
        im = matrix_panel(ax, mats[name], T, sh, kind, vmax=vmax)
        if i == len(names) - 1:
            ax.set_xlabel("lead time [h]")
        else:
            ax.set_xticklabels([])
        ax.set_ylabel("lead time [h]")
        if i == 0:
            ax.set_title(r"correlation $\rho$" if kind == "corr"
                         else r"covariance $\Sigma$")

    label = ("correlation" if kind == "corr" else f"covariance [{first['units']}$^2$]")
    fig.colorbar(im, ax=axes[:, 1].tolist(), fraction=0.035, pad=0.02, label=label)
    fig.suptitle(_title(first, f", test window {first['index']}"))
    return fig

### Figure 2 -- sample paths and the mean

The most direct read of what a covariance *means*: draws for one window, next to the mean and
the truth. A diagonal-plus-rank-1 covariance produces white noise around the mean plus one
shared up/down shift; a covariance with a decaying autocorrelation produces trajectories that
look like the series itself. Every panel shares one $y$ range, so the arms are comparable by
eye, and the marginal band is drawn underneath as the reference the draws should fill.

In [ ]:
def plot_sample_paths(windows, n_show=None, ncols=None, aspect=0.30, band=True):
    """Figure 2: draws, the predictive mean and the truth, one panel per arm.

    `ncols` wraps the arms over several rows. A long horizon needs it: 720 steps in the
    1.5-inch panel of a four-across row is a solid blur, and it is the *shape* of a single
    path that the figure exists to show.
    """
    names = list(windows)
    first = windows[names[0]]
    T, sh = first["target"].shape[0], first["step_hours"]
    t, ticks = lead_axis(T, sh)
    n_show = N_SHOW if n_show is None else n_show
    ncols = len(names) if ncols is None else ncols
    nrows = int(np.ceil(len(names) / ncols))

    lo = min(float(np.min(w["draws"])) for w in windows.values())
    hi = max(float(np.max(w["draws"])) for w in windows.values())
    lo = min(lo, min(float(w["target"].min()) for w in windows.values()))
    hi = max(hi, max(float(w["target"].max()) for w in windows.values()))
    pad = 0.05 * (hi - lo)

    fig, axes = pc.subplots(nrows, ncols, aspect=aspect, squeeze=False, sharey=True)
    for ax in axes.ravel()[len(names):]:
        ax.remove()
    for i, name in enumerate(names):
        ax, w = axes.ravel()[i], windows[name]
        color = w["arm"]["color"]
        if band:
            ax.fill_between(t, w["mean"] - 2 * w["std"], w["mean"] + 2 * w["std"],
                            color=color, alpha=0.15, lw=0)
        ax.plot(t, w["draws"][:, :n_show], color=color, lw=0.5, alpha=0.7)
        ax.plot(t, w["target"], color=pc.COLORS["signal"], lw=PLOT.thin_line_width,
                label="truth")
        ax.plot(t, w["mean"], color=color, lw=PLOT.line_width, label="mean")
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xticks(ticks)
        ax.set_title(w["arm"]["label"])
        if i // ncols == nrows - 1:
            ax.set_xlabel("lead time [h]")
        if i % ncols == 0:
            ax.set_ylabel(f"[{w['units']}]")
        if i == 0:
            ax.legend(loc="upper left", handlelength=1.4)
    shown = min(n_show, min(w["draws"].shape[1] for w in windows.values()))
    fig.suptitle(_title(first, f", test window {first['index']}: {shown} draws"))
    return fig

### Figure 3 -- the covariance comparison, model over empirical

The comparison the single window cannot make, in two rows:

* **top** -- each arm's **test-set mean** $\overline{\Sigma}$. Every arm's covariance is
  input-dependent, so the structure has to be read off an average over cases rather than off
  one case;
* **bottom** -- the **empirical** covariance of that same arm's residuals, $y - \mu_{\rm arm}$,
  over the same windows. Each column is therefore a self-contained "predicted against
  realised" pair, with no mean borrowed from another arm.

Shown as correlation by default (`kind="corr"`): the arms differ in their marginal variance
profile too, and dividing it out isolates the dependence structure, which is the part the
parameterization is responsible for. The annotation is $\overline{|\Delta\rho|}$, the mean
absolute off-diagonal gap between the two panels of that column -- one number for "how close
is this arm's dependence to its own realised dependence".

One caveat on the bottom row: the test windows **overlap** -- consecutive windows share all
but one step of their horizon -- so its effective sample size is far below the window count,
especially at $T = 192$ where 2689 overlapping windows estimate a $192 \times 192$ matrix. It
is a reference, not ground truth.

In [ ]:
def plot_cov_comparison(scans, kind="corr", annotate=True, aspect=0.56):
    """Figure 3: per-arm model covariance (top) over its own empirical residual one (bottom)."""
    names = list(scans)
    first = scans[names[0]]
    T, sh = first["mean_cov"].shape[0], first["step_hours"]

    conv = cov_to_corr if kind == "corr" else (lambda m: m)
    model = [conv(scans[n]["mean_cov"]) for n in names]
    empir = [conv(scans[n]["emp_cov"]) for n in names]
    vmax = offdiag_absmax([*model, *empir])
    off = ~np.eye(T, dtype=bool)

    fig, axes = pc.subplots(2, len(names), aspect=aspect, squeeze=False)
    for i, name in enumerate(names):
        for row, mats, tag in ((0, model, r"model $\overline{\Sigma}$"),
                               (1, empir, "empirical (own mean)")):
            ax = axes[row, i]
            im = matrix_panel(ax, mats[i], T, sh, kind, vmax=vmax)
            if row == 0:
                ax.set_title(scans[name]["arm"]["label"])
                ax.set_xticklabels([])
            else:
                ax.set_xlabel("lead [h]")
            if i == 0:
                ax.set_ylabel(f"{tag}\nlead [h]")
            else:
                ax.set_yticklabels([])
        if annotate:
            gap = float(np.nanmean(np.abs(model[i] - empir[i])[off]))
            sym = r"\rho" if kind == "corr" else r"\Sigma"
            axes[0, i].annotate(rf"$\overline{{|\Delta {sym}|}} = {gap:.3f}$",
                                xy=(0.04, 0.94), xycoords="axes fraction",
                                fontsize=PLOT.annotation_size, va="top",
                                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                                          ec="0.7", lw=0.5, alpha=0.85))

    label = ("correlation" if kind == "corr" else f"covariance [{first['units']}$^2$]")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.01, label=label)
    fig.suptitle(_title(first, f": model vs empirical ({first['n']} windows)"))
    return fig

### Figure 4 -- autocorrelation, predicted against realised

Both curves in a colour are the same arm: the **solid** line is its predicted $\rho(h)$, the
lag average of $\overline{\Sigma}$'s correlation matrix, and the **dashed** line is the
empirical $\rho(h)$ of *its own* residuals over the same windows. So every arm is judged
against the realised dependence left by its own mean, and the vertical gap inside one colour
is that arm's error -- no reference borrowed from another arm.

The dashed curves nearly coincide across arms, which is what keeps the solid ones comparable
with each other: the four arms leave almost the same residual dependence behind, so the spread
between the predictions is a difference between models. Watch for the exception -- an arm whose
dashed line separates from the others has a *mean* that differs, not a covariance, and the
NLL-fitted arms are where that shows.

Lags run to `LAG_MAX_FRAC` of the horizon; the offset-$h$ diagonal averages $T-h$ entries, so
the far tail is a couple of numbers wide and is cut rather than drawn. $\rho(0) = 1$ on both
sides by construction.

In [ ]:
def _outside_legend(fig, handles, loc="outside right center"):
    """One legend, always outside the axes -- these panels are dense and a legend inside sits
    on exactly the decaying curves it would explain."""
    fig.legend(handles=handles, loc=loc, handlelength=1.6, frameon=False,
               fontsize=PLOT.legend_size)


def plot_autocorrelation(scans, aspect=0.34):
    """Figure 4: predicted ACF (solid) against each arm's own empirical ACF (dashed)."""
    names = list(scans)
    first = scans[names[0]]
    T, sh = first["mean_cov"].shape[0], first["step_hours"]
    lags, ticks, n = lag_axis(T, sh)

    fig, ax = pc.subplots(1, 1, aspect=aspect)
    for name in names:
        s = scans[name]
        color = s["arm"]["color"]
        ax.plot(lags, lag_average(cov_to_corr(s["mean_cov"]))[:n],
                color=color, lw=PLOT.line_width)
        ax.plot(lags, lag_average(cov_to_corr(s["emp_cov"]))[:n],
                color=color, lw=PLOT.thin_line_width, ls="--")
    ax.axhline(0.0, color=pc.COLORS["guide"], lw=PLOT.thin_line_width)
    ax.set_xticks(ticks)
    ax.set_xlabel("lag [h]")
    ax.set_ylabel(r"$\rho(h)$")
    ax.set_title("autocorrelation: predicted (solid) vs own residuals (dashed)")

    # Colour carries the arm and dash carries model-or-empirical, so the legend says both
    # once rather than listing eight lines.
    handles = [Line2D([], [], color=scans[k]["arm"]["color"], lw=PLOT.line_width,
                      label=scans[k]["arm"]["label"]) for k in names]
    handles += [
        Line2D([], [], color="0.35", lw=PLOT.line_width, label="predicted"),
        Line2D([], [], color="0.35", lw=PLOT.thin_line_width, ls="--",
               label="empirical (own residuals)"),
    ]
    _outside_legend(fig, handles)
    fig.suptitle(_title(first, f", {first['n']} windows"))
    return fig

### Figure 5 -- the ACF ratio

$\hat\rho(h) / \rho_{\rm model}(h)$, the ratio of the two curves of figure 4 taken inside each
colour. Above 1 the arm **under**-states the dependence at that lag, below 1 it
**over**-states it, and 1 is correct. It is on a **log** scale, because a ratio is
multiplicative: "twice too little" and "twice too much" should be the same distance from the
line.

$\rho(0) = 1$ on both sides, so the ratio starts at lag 1. It is **blanked wherever the
predicted correlation falls below `RATIO_MIN_RHO`**: there the denominator is ~0, and a spike
at $\rho_{\rm model}\approx 0$ says nothing about the fit. That blanking is not cosmetic at
$T = 192$ -- an arm predicting near-zero correlation at almost every lag has *no* well-defined
ratio, and would otherwise appear as a few disconnected fragments at the top of the panel or
vanish with no explanation. The coverage per arm is printed, and an arm that vanishes entirely
is named inside the panel; read it off figure 4 instead, where a flat $\rho \approx 0$ is
visible as exactly that.

In [ ]:
def plot_acf_ratio(scans, aspect=0.34, ratio_ylim=(0.2, 8.0)):
    """Figure 5: empirical-over-predicted ACF, each arm against its own residuals."""
    names = list(scans)
    first = scans[names[0]]
    T, sh = first["mean_cov"].shape[0], first["step_hours"]
    lags, ticks, n = lag_axis(T, sh)

    fig, ax = pc.subplots(1, 1, aspect=aspect)
    coverage = {}
    for name in names:
        s = scans[name]
        model = lag_average(cov_to_corr(s["mean_cov"]))[:n]
        own = lag_average(cov_to_corr(s["emp_cov"]))[:n]
        keep = np.abs(model) > RATIO_MIN_RHO
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(keep, own / model, np.nan)
        # Coverage over the *drawn* lags (1..n-1); lag 0 is 1 by construction and excluded.
        coverage[name] = float(keep[1:].mean())
        ax.plot(lags[1:], ratio[1:], color=s["arm"]["color"], lw=PLOT.line_width,
                label=s["arm"]["label"])
    ax.axhline(1.0, color=pc.COLORS["guide"], lw=PLOT.thin_line_width, ls=":")
    ax.set_xticks(ticks)
    ax.set_yscale("log")            # a ratio is multiplicative: over and under, symmetric
    ax.set_ylim(*ratio_ylim)
    # A log axis labels its *minor* ticks too ("3 x 10^-1", ...), which on a short panel is
    # unreadable clutter; the ratio only needs the doubling ladder.
    ax.yaxis.set_major_locator(mticker.FixedLocator([0.25, 0.5, 1.0, 2.0, 4.0, 8.0]))
    ax.yaxis.set_major_formatter(
        mticker.FixedFormatter(["0.25", "0.5", "1", "2", "4", "8"]))
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.set_xlabel("lag [h]")
    ax.set_ylabel(r"$\hat\rho(h)\,/\,\rho_{\mathrm{model}}(h)$")
    ax.set_title(r"ACF ratio ($>1$: under-states dependence)")

    # An arm whose predicted correlation is below RATIO_MIN_RHO at (almost) every lag has no
    # ratio to draw. Saying so is the difference between "this arm is missing from the panel"
    # and "this arm predicts no correlation", which is a finding.
    print(f"  ACF ratio, lags with rho_model > {RATIO_MIN_RHO}:")
    for k, v in coverage.items():
        print(f"    {k:24s} {v:6.1%}" + ("   <- no ratio drawn" if v == 0 else ""))
    gone = [scans[k]["arm"]["label"] for k, v in coverage.items() if v == 0]
    if gone:
        ax.annotate("no ratio defined: " + ", ".join(gone),
                    xy=(0.03, 0.03), xycoords="axes fraction", ha="left", va="bottom",
                    fontsize=PLOT.annotation_size, color="0.35",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7",
                              lw=0.5, alpha=0.85))

    handles = [Line2D([], [], color=scans[k]["arm"]["color"], lw=PLOT.line_width,
                      label=scans[k]["arm"]["label"]) for k in names]
    _outside_legend(fig, handles)
    fig.suptitle(_title(first, f", {first['n']} windows"))
    return fig

## 4 $T = 96$, arms as trained

96 steps of the dataset's own cadence: 4 days on ETTh (hourly), 24 h on ETTm (15-minute).
2785 test windows on either ETTh.

**The objective is not held fixed in this section, on purpose.** The two explicit-density
Gaussians are fitted by maximum likelihood, which is the strongest thing available to a full
covariance and the objective they exist to exploit; the sampler has no density at all
(`SampleBased` rejects `loss=nll` outright), so a scoring rule is the only option SB has; and
SCORE uses the two-stage fit -- stage 1 fits the marginals $D$ with CRPS, stage 2 freezes them
and fits the core $s$ with the kernel score. So each column shows a parameterization at its
own best. Sections 6-7 repeat all five figures with every arm at the kernel score, which is
where to look for a matched-objective reading.

One consequence to keep in mind while reading: a kernel-score-fitted Chol is *not*
density-calibrated -- its reported log score is unusable, and on `etth2/T96` finite nonsense --
so the two objectives cannot be mixed inside one *score* comparison, only inside a structural
one like this.

In [ ]:
W96, S96 = prepare(DATASET, T_SHORT, ARMS_ASTRAINED)

In [ ]:
fig = plot_predictive_and_cov(W96, kind="cov")
save(fig, f"T{T_SHORT}_astrained_predictive_cov")
plt.show()

In [ ]:
fig = plot_sample_paths(W96)
save(fig, f"T{T_SHORT}_astrained_sample_paths")
plt.show()

In [ ]:
fig = plot_cov_comparison(S96, kind="corr")
save(fig, f"T{T_SHORT}_astrained_cov_comparison")
plt.show()

In [ ]:
fig = plot_autocorrelation(S96)
save(fig, f"T{T_SHORT}_astrained_autocorrelation")
plt.show()

In [ ]:
fig = plot_acf_ratio(S96)
save(fig, f"T{T_SHORT}_astrained_acf_ratio")
plt.show()

## 5 $T = 192$, arms as trained

192 steps: 8 days on ETTh, 48 h on ETTm. 2689 test windows on either ETTh -- the same
benchmark, the same forecast origin, an event dimension twice as large. Three things change with
the horizon and are worth reading the figures for:

* Chol now emits **18,528** numbers per case against the two structured arms' 384
  (4,656 against 192 at $T = 96$), so the unstructured head grows quadratically while SCORE
  grows linearly;
* SB's 100-member protocol covariance is **rank-99** against a $192 \times 192$
  target, i.e. rank-deficient for the first time (the single-window panels use `N_DRAW`
  members instead, for the reason given in section 1);
* the diurnal cycle sits at DFT bin $k = T\,\Delta t / 24\,\mathrm{h}$, which on ETTh is
  $k = 8$ here against $k = 4$ at $T = 96$ -- twice the frequency resolution around the one
  periodicity the series actually has. (On ETTm it is $k = 1$ at $T = 96$ and $k = 2$ at
  $T = 192$: the horizon spans one and two diurnal cycles.)


In [ ]:
W192, S192 = prepare(DATASET, T_LONG, ARMS_ASTRAINED)

In [ ]:
fig = plot_predictive_and_cov(W192, kind="cov")
save(fig, f"T{T_LONG}_astrained_predictive_cov")
plt.show()

In [ ]:
# Two across rather than four: at T=192 a four-across row gives each panel 1.5 in for 192
# steps, and the individual paths stop being separable.
fig = plot_sample_paths(W192, n_show=6, ncols=2, aspect=0.62)
save(fig, f"T{T_LONG}_astrained_sample_paths")
plt.show()

In [ ]:
fig = plot_cov_comparison(S192, kind="corr")
save(fig, f"T{T_LONG}_astrained_cov_comparison")
plt.show()

In [ ]:
fig = plot_autocorrelation(S192)
save(fig, f"T{T_LONG}_astrained_autocorrelation")
plt.show()

In [ ]:
fig = plot_acf_ratio(S192)
save(fig, f"T{T_LONG}_astrained_acf_ratio")
plt.show()

## 6 $T = 96$, every arm at the kernel score

The same four parameterizations and the same window as section 4, with the two Gaussians
swapped from their NLL checkpoints to their kernel-score ones (`phys_lora_r1_kernel_score`,
`phys_cholesky_kernel_score`). SB and SCORE are the *identical* checkpoints as in
section 4 -- neither has ever been NLL-fitted -- so they are the fixed point the other two move
against, and a change in their panels between the two sections would be a bug rather than a
finding.

What this section isolates is the objective's effect on the *learned dependence* at a fixed
parameterization. The two readings answer different questions and neither supersedes the
other: section 4 asks what each family does when trained the way it is meant to be, this one
asks what the objective itself does -- and a comparison *between parameterizations* is only
clean when the loss is the same everywhere, which is here.

In [ ]:
W96K, S96K = prepare(DATASET, T_SHORT, ARMS_KERNEL_SCORE)

In [ ]:
fig = plot_predictive_and_cov(W96K, kind="cov")
save(fig, f"T{T_SHORT}_ks_predictive_cov")
plt.show()

In [ ]:
fig = plot_sample_paths(W96K)
save(fig, f"T{T_SHORT}_ks_sample_paths")
plt.show()

In [ ]:
fig = plot_cov_comparison(S96K, kind="corr")
save(fig, f"T{T_SHORT}_ks_cov_comparison")
plt.show()

In [ ]:
fig = plot_autocorrelation(S96K)
save(fig, f"T{T_SHORT}_ks_autocorrelation")
plt.show()

In [ ]:
fig = plot_acf_ratio(S96K)
save(fig, f"T{T_SHORT}_ks_acf_ratio")
plt.show()

## 7 $T = 192$, every arm at the kernel score

The long horizon at matched objective. This is where Chol is under the most
pressure: 18,528 free numbers per case, fitted by a scoring rule that -- unlike the log score
-- never asks for a density, so nothing in training constrains the directions of $\Sigma$ the
kernel score happens to be insensitive to at its bandwidth.


In [ ]:
W192K, S192K = prepare(DATASET, T_LONG, ARMS_KERNEL_SCORE)

In [ ]:
fig = plot_predictive_and_cov(W192K, kind="cov")
save(fig, f"T{T_LONG}_ks_predictive_cov")
plt.show()

In [ ]:
# Two across rather than four: at T=192 a four-across row gives each panel 1.5 in for 192
# steps, and the individual paths stop being separable.
fig = plot_sample_paths(W192K, n_show=6, ncols=2, aspect=0.62)
save(fig, f"T{T_LONG}_ks_sample_paths")
plt.show()

In [ ]:
fig = plot_cov_comparison(S192K, kind="corr")
save(fig, f"T{T_LONG}_ks_cov_comparison")
plt.show()

In [ ]:
fig = plot_autocorrelation(S192K)
save(fig, f"T{T_LONG}_ks_autocorrelation")
plt.show()

In [ ]:
fig = plot_acf_ratio(S192K)
save(fig, f"T{T_LONG}_ks_acf_ratio")
plt.show()

## 8 Paper figure -- ETTh2 next to the EUPPBench figure

The time-series companion of `euppbench_truth_correlation.pdf`, built to sit beside it: the
same single-column width and height (the height read off that PDF, so the two cannot drift
apart), panels and colorbar on the same rows, the same boxed panel tags, and the same
comparison -- LorD $r{=}1$ against SCORE, both fitted with the kernel score, which is the
pair the EUPPBench main figure shows. Fixed to **ETTh2, univariate, $T = 96$**, whatever
`DATASET` says above.

* **(a)** one test window of SCORE: the last 72 h of input, the truth, the predictive mean
  (dashed, drawn over the truth) and the $\pm 2\sigma$ marginal band.
* **(b), (c)** the correlation over the horizon, split along the diagonal: above it the
  arm's **test-set mean** $\overline{\Sigma}$ as a correlation matrix, below it the empirical
  correlation of that arm's own residuals -- the figure 3 comparison (section 3), folded into
  one panel per arm so it fits the column. The panels take the EUPPBench map panels' size, so
  they are slightly stretched vertically; the matrices are $96 \times 96$ either way. The
  colormap is sequential (`PAPER_CMAP`) because the diverging red-blue is already the
  EUPPBench figure's temperature scale.

The dark top-left corner of (c), partly under its panel tag, is not a fitting artefact: the
core is circulant, so $\rho(1, 96) = \rho(1, 2)$ holds exactly for every window, while the
residuals put much less correlation there (the cell below prints both).

Unlike the diagnostic figures above, this one is saved whether `SAVE` is on or not -- it
*is* the paper figure, so re-running the section is how it gets refreshed.

In [ ]:
#: The paper figure's task and arms: ETTh2 whatever `DATASET` is (the seed is still `SEED`),
#: and the kernel-score arms, as in the EUPPBench main figure.
PAPER_DATASET, PAPER_T, PAPER_WINDOW = "etth2", 96, 400
PAPER_ARMS = {"LorD": ARMS_KERNEL_SCORE["LorD (r=1, KS)"],
              "SCORE": ARMS_KERNEL_SCORE["SCORE (KS)"]}

paper_window = analyze_window(PAPER_DATASET, PAPER_T, PAPER_ARMS["SCORE"], index=PAPER_WINDOW)
paper_scans = {name: scan_test(PAPER_DATASET, PAPER_T, arm) for name, arm in PAPER_ARMS.items()}

for name, s in paper_scans.items():
    model, emp = cov_to_corr(s["mean_cov"]), cov_to_corr(s["emp_cov"])
    print(f"{name:5s} n={s['n']}  RMSE={s['rmse']:.3f} {s['units']}  "
          f"rho(1,2): model {model[0, 1]:.2f} / empirical {emp[0, 1]:.2f}   "
          f"rho(1,{PAPER_T}): model {model[0, -1]:.2f} / empirical {emp[0, -1]:.2f}")

In [ ]:
import re

from matplotlib.layout_engine import PlaceHolderLayoutEngine

#: Hours of input drawn left of the forecast origin in panel (a).
PAPER_CONTEXT_H = 72
EUPP_FIGURE = pc.FIG_DIR / "euppbench_truth_correlation.pdf"
#: The EUPPBench figure's geometry, measured off its saved PDF (inches; "top" is the distance
#: below the canvas's top edge): its panel frames and its horizontal colorbar. The panels and
#: the colorbar here take the same rows, and the correlation panels the maps' size, so the two
#: figures line up side by side. Re-measure if that figure's layout changes.
EUPP_PANEL_TOP, EUPP_PANEL_H, EUPP_PANEL_W, EUPP_PANEL_GAP = 0.10, 0.973, 0.826, 0.097
EUPP_CBAR_TOP, EUPP_CBAR_H = 1.147, 0.056
#: Sequential, not the diverging ramp the rest of this notebook uses for correlation: the
#: EUPPBench figure next to this one already spends the red-blue `vik` on temperature, and every
#: correlation drawn here is positive. Reversed `lipari` is crameri's magma, dark = high; the
#: same colormap as the double-column overview in analyze_euppbench_predictions.ipynb.
try:
    from cmap import Colormap
    PAPER_CMAP = Colormap("crameri:lipari").to_mpl().reversed()
except Exception:                     # `cmap` is optional in plot_config as well
    PAPER_CMAP = plt.get_cmap("magma_r")


def pdf_height_in(path, fallback=113.292 / 72):
    """Height of a saved PDF in inches, from its MediaBox (matplotlib writes it uncompressed).

    The EUPPBench figure's height comes out of its map-grid geometry, so copying the number
    here would go stale the next time that layout changes.
    """
    try:
        box = re.search(rb"/MediaBox\s*\[\s*([-\d.]+)\s+([-\d.]+)\s+([-\d.]+)\s+([-\d.]+)\s*\]",
                        Path(path).read_bytes())
    except OSError:
        box = None
    if box is None:
        print(f"warning: no MediaBox read from {path}; using {fallback:.4f} in")
        return fallback
    return (float(box.group(4)) - float(box.group(2))) / 72.0


def panel_tag(ax, text, loc=(0.025, 0.975)):
    """A caption-referable `(a)` / `(b)` marker inside the panel -- the EUPPBench notebook's."""
    ax.text(*loc, text, transform=ax.transAxes, va="top", ha="left",
            fontsize=PLOT.annotation_size, zorder=6,
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white",
                      edgecolor="0.7", linewidth=PLOT.axes_line_width, alpha=0.85))


def split_triangle(upper, lower):
    """`upper` above the diagonal and `lower` below it, as drawn with `origin="lower"`."""
    i, j = np.indices(upper.shape)
    return np.where(i > j, upper, np.where(i < j, lower, 1.0))


def ink_on(cmap, value):
    """Black or white, whichever reads on `cmap`'s color at `value` (0..1)."""
    r, g, b, _ = cmap(float(value))
    return "0.15" if 0.2126 * r + 0.7152 * g + 0.0722 * b > 0.45 else "white"


def plot_paper_forecast_correlation(window, scans, context_h=PAPER_CONTEXT_H, cmap=None,
                                    height=None):
    """Panel (a) the forecast of `window`, then one split correlation panel per scan.

    Placed in inches rather than by constrained layout, so that the panels and the colorbar
    sit exactly on the EUPPBench figure's rows; panel (a) takes whatever width the two
    map-sized correlation panels leave.
    """
    cmap = PAPER_CMAP if cmap is None else cmap
    width = pc.width_of("single")
    height = pdf_height_in(EUPP_FIGURE) if height is None else height
    left, gap_ab, right = 0.20, 0.17, 0.02                          # inches
    x_c = width - right - EUPP_PANEL_W
    x_b = x_c - EUPP_PANEL_GAP - EUPP_PANEL_W
    y0 = height - EUPP_PANEL_TOP - EUPP_PANEL_H

    # A placeholder engine rather than layout="none": savefig restores a None engine by
    # re-reading rcParams, where `pc.apply_style` switched constrained layout on, so a
    # layout="none" figure comes back from `pc.save` with constrained layout attached.
    fig = plt.figure(figsize=(width, height), layout=PlaceHolderLayoutEngine(True, True))
    def rect(x, w, y=y0, h=EUPP_PANEL_H):
        return [x / width, y / height, w / width, h / height]
    ax_f = fig.add_axes(rect(left, x_b - gap_ab - left))
    ax_m = [fig.add_axes(rect(x, EUPP_PANEL_W)) for x in (x_b, x_c)]
    cax = fig.add_axes(rect(x_b, width - right - x_b,
                            y=height - EUPP_CBAR_TOP - EUPP_CBAR_H, h=EUPP_CBAR_H))

    # (a) the forecast: band, input, truth, and the mean dashed on top of the truth. The
    # ticks carry the unit, so there is no axis label.
    color, dt = window["arm"]["color"], window["step_hours"]
    T = len(window["target"])
    lead = np.arange(1, T + 1) * dt
    n_ctx = int(round(context_h / dt))
    mu, sd = window["mean"], window["std"]
    ax_f.fill_between(lead, mu - 2 * sd, mu + 2 * sd, color=color, alpha=PLOT.band_alpha,
                      lw=0, zorder=1)
    ax_f.plot(np.arange(-n_ctx + 1, 1) * dt, window["context"][-n_ctx:],
              color=pc.COLORS["guide"], lw=0.8, zorder=2)
    ax_f.plot(lead, window["target"], color="k", lw=0.8, zorder=3)
    ax_f.plot(lead, mu, color=color, lw=1.0, ls=(0, (3, 1.5)), zorder=4)
    ax_f.axvline(0, color=pc.COLORS["guide"], lw=0.5, ls=":", zorder=0)
    ax_f.set_xlim(-n_ctx * dt, T * dt)
    hours = [-48, 0, 48, 96]
    ax_f.set_xticks(hours, labels=[f"${h:d}$h" for h in hours])
    panel_tag(ax_f, f"(a) {DATASET_LABEL[PAPER_DATASET]}")

    # (b), (c) model above the diagonal, the arm's own residuals below it, on one scale that
    # starts at 0 unless some correlation is negative.
    imgs = [split_triangle(cov_to_corr(s["mean_cov"]), cov_to_corr(s["emp_cov"]))
            for s in scans.values()]
    vmin = min(0.0, np.floor(10 * min(np.nanmin(im) for im in imgs)) / 10)
    norm = plt.Normalize(vmin, 1.0)
    angle = np.degrees(np.arctan2(EUPP_PANEL_H, EUPP_PANEL_W))   # the diagonal, on screen
    for k, (ax, name, img) in enumerate(zip(ax_m, scans, imgs)):
        n = img.shape[0]
        im = ax.imshow(img, origin="lower", cmap=cmap, norm=norm, aspect="auto",
                       extent=(0.5, n + 0.5, 0.5, n + 0.5), interpolation="nearest",
                       rasterized=True)
        ax.plot([0.5, n + 0.5], [0.5, n + 0.5], color=ink_on(cmap, 1.0), lw=0.3)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        panel_tag(ax, f"({chr(ord('b') + k)}) {name}")
        i, j = np.indices(img.shape)
        for x, y, text, side in [(0.34, 0.62, "Model", i > j), (0.66, 0.38, "Empirical", i < j)]:
            ax.text(x, y, text, transform=ax.transAxes, rotation=angle, ha="center",
                    va="center", fontsize=PLOT.annotation_size - 1,
                    color=ink_on(cmap, norm(np.nanmedian(img[side]))))

    cb = fig.colorbar(im, cax=cax, orientation="horizontal",
                      format=mticker.FuncFormatter(lambda v, _: f"${v:g}$"))   # math minus
    cb.set_ticks(np.round(np.arange(vmin, 1.0 + 1e-9, 0.2), 1))
    cb.set_label(r"correlation $\rho$", size=PLOT.colorbar_label_size)
    return fig


fig = plot_paper_forecast_correlation(paper_window, paper_scans)
print("saved:", pc.save(fig, f"ts_{PAPER_DATASET}_forecast_correlation.pdf"))
plt.show()